In [114]:
import warnings
warnings.simplefilter(action='ignore',category=FutureWarning)


from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
from sklearn.model_selection import (
    train_test_split,
    cross_val_score
) 
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
df = pd.read_csv("Telco_churn_data.xls")
df.head()
df.isnull().sum()
df.describe()
df.nunique()
df.head()
yes_no_columns = [
    "Partner",
    "Dependents",
    "PhoneService",
    "PaperlessBilling",
    "Churn"
]
df[yes_no_columns]= df[yes_no_columns].replace({
    "Yes":1,
    "No":0
})
df["gender"] = df["gender"].replace({
    "Male":1,
    "Female":0
})
df.drop(columns="customerID",inplace = True)
df = pd.get_dummies(
    df,
    columns = [
        "MultipleLines",
        "InternetService",
        "OnlineSecurity",
        "OnlineBackup",
        "DeviceProtection",
        "TechSupport",
        "StreamingTV",
        "StreamingMovies",
        "PaymentMethod",
        "Contract"
    ],
    drop_first=True
)
pd.to_numeric(df["TotalCharges"],errors='coerce')
mask = pd.to_numeric(df["TotalCharges"],errors='coerce').isna()
df = df[~mask]
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"])
#line below is meant to address skewness
df["TotalCharges"] = np.log1p(df["TotalCharges"]) 
cols_to_scale = ["tenure","MonthlyCharges","TotalCharges"]
scalr = StandardScaler()
df[cols_to_scale] = scalr.fit_transform(df[cols_to_scale])
df["TotalCharges"].head()

0   -2.272642
1    0.388798
2   -1.455489
3    0.371903
4   -1.238572
Name: TotalCharges, dtype: float64

In [111]:
df["PhoneService"].info()

<class 'pandas.core.series.Series'>
Index: 7032 entries, 0 to 7042
Series name: PhoneService
Non-Null Count  Dtype
--------------  -----
7032 non-null   int64
dtypes: int64(1)
memory usage: 109.9 KB


In [117]:
X = df.drop(columns=["Churn"],axis=1)
Y=df["Churn"]
X_train,X_test,Y_train,Y_test = train_test_split(
    X,
    Y,
    test_size=0.2,
    random_state=42
)
model = LogisticRegression()
model.fit(X_train,Y_train)
prediction = model.predict(X_test)
accuracy = accuracy_score(Y_test,prediction)
cm = confusion_matrix(Y_test,prediction)
cr = classification_report(Y_test,prediction)
print(accuracy)
print(cm)
print(cr)

0.7945984363894811
[[931 102]
 [187 187]]
              precision    recall  f1-score   support

           0       0.83      0.90      0.87      1033
           1       0.65      0.50      0.56       374

    accuracy                           0.79      1407
   macro avg       0.74      0.70      0.71      1407
weighted avg       0.78      0.79      0.79      1407



In [113]:
for col in df.columns:
    if df[col].dtype == "object":
        print(col)
        print(df[col].unique())
        print()

MultipleLines
['No phone service' 'No' 'Yes']

